In [1]:
import sys
import json
from pathlib import Path
from collections import defaultdict
from datetime import datetime

from IPython.display import display, Markdown

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from config.settings import (
    FILTER_MANIFEST_PATH,
    TEXT_CHUNKS_PATH,
    CHUNK_IMAGE_LINKS_PATH,
    LINKED_DIR,
)

LINKED_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
assert FILTER_MANIFEST_PATH.exists(), f"Not found: {FILTER_MANIFEST_PATH}"
assert TEXT_CHUNKS_PATH.exists(), f"Not found: {TEXT_CHUNKS_PATH}"

filter_manifest = json.loads(FILTER_MANIFEST_PATH.read_text(encoding="utf-8"))
chunks = json.loads(TEXT_CHUNKS_PATH.read_text(encoding="utf-8"))

kept_images = [img for img in filter_manifest if img.get("decision") == "keep"]

display(Markdown(f"""
### Inputs Loaded

| Item | Value |
|------|-------|
| **Text chunks** | {len(chunks)} |
| **Kept images** | {len(kept_images)} |
""".strip()))

### Inputs Loaded

| Item | Value |
|------|-------|
| **Text chunks** | 1391 |
| **Kept images** | 866 |

In [3]:
page_to_images = defaultdict(list)

for img in kept_images:
    page_to_images[img["page"]].append({
        "filename": img["filename"],
        "page": img["page"],
        "width": img["width"],
        "height": img["height"],
        "file_size_bytes": img["file_size_bytes"],
    })

pages_with_images = len(page_to_images)

display(Markdown(f"""
### Image Page Index

| Item | Value |
|------|-------|
| **Kept images** | {len(kept_images)} |
| **Pages with images** | {pages_with_images} |
""".strip()))

### Image Page Index

| Item | Value |
|------|-------|
| **Kept images** | 866 |
| **Pages with images** | 536 |

In [4]:
linked_records = []
all_linked_filenames = set()

for chunk in chunks:
    matched_images = []
    for pg in chunk["source_pages"]:
        matched_images.extend(page_to_images.get(pg, []))

    for img in matched_images:
        all_linked_filenames.add(img["filename"])

    linked_records.append({
        "chunk_id": chunk["chunk_id"],
        "section_title": chunk["section_title"],
        "section_path": chunk["section_path"],
        "page_start": chunk["page_start"],
        "page_end": chunk["page_end"],
        "source_pages": chunk["source_pages"],
        "image_count": len(matched_images),
        "images": matched_images,
    })

chunks_with_images = sum(1 for r in linked_records if r["image_count"] > 0)
total_linked = sum(r["image_count"] for r in linked_records)
unlinked_images = [img for img in kept_images if img["filename"] not in all_linked_filenames]

display(Markdown(f"### Linking Complete: {chunks_with_images} chunks have images"))

### Linking Complete: 423 chunks have images

In [5]:
CHUNK_IMAGE_LINKS_PATH.write_text(
    json.dumps(linked_records, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

# Sample linked chunks (up to 3)
samples = [r for r in linked_records if r["image_count"] > 0][:3]
sample_blocks = ""
for s in samples:
    img_rows = "\n".join(
        f"  - `{img['filename']}` ({img['width']}x{img['height']}, {img['file_size_bytes']/1024:.1f} KB)"
        for img in s["images"][:5]
    )
    extra = f"\n  - ... and {len(s['images'])-5} more" if len(s["images"]) > 5 else ""
    sample_blocks += f"""
---
**{s['chunk_id']}** | Pages {s['page_start']}-{s['page_end']} | {s['image_count']} images

Path: `{' > '.join(s['section_path'])}`

Images:
{img_rows}{extra}
"""

display(Markdown(f"""
### Image-Text Linking Summary

| Item | Value |
|------|-------|
| **Total chunks** | {len(linked_records)} |
| **Kept images** | {len(kept_images)} |
| **Chunks with images** | {chunks_with_images} |
| **Total image links** | {total_linked} |
| **Unlinked kept images** | {len(unlinked_images)} |
| **Output** | `{CHUNK_IMAGE_LINKS_PATH.relative_to(project_root)}` |
| **Timestamp** | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} |

### Sample Linked Chunks
{sample_blocks}
""".strip()))

### Image-Text Linking Summary

| Item | Value |
|------|-------|
| **Total chunks** | 1391 |
| **Kept images** | 866 |
| **Chunks with images** | 423 |
| **Total image links** | 1282 |
| **Unlinked kept images** | 0 |
| **Output** | `SCADA-DIP\data\linked\chunk_image_links.json` |
| **Timestamp** | 2026-04-24 09:36:56 |

### Sample Linked Chunks

---
**chunk_000016** | Pages 48-48 | 1 images

Path: `Plan > Components and single-site architectures > Components overview`

Images:
  - `page0048_img001_w1278_h0576.png` (1278x576, 299.5 KB)

---
**chunk_000017** | Pages 48-48 | 1 images

Path: `Plan > Components and single-site architectures > Time synchronization`

Images:
  - `page0048_img001_w1278_h0576.png` (1278x576, 299.5 KB)

---
**chunk_000018** | Pages 49-49 | 1 images

Path: `Plan > Components and single-site architectures > Power SCADA Server component`

Images:
  - `page0049_img002_w1270_h0572.png` (1270x572, 228.6 KB)